# 04 — OCR разворотов и подготовка anchors

## Цель

Автоматически обработать каждый разворот из `data/besy/pages/` без ручной разметки: найти исходные OCR-боксы, вычислить границу между страницами, создать перекрывающиеся вырезы, применить UVDoc dewarping и повторно распознать обе страницы.

Matching с FB2 и создание anchors намеренно **не выполняются** в этом ноутбуке. Его результат — проверяемые OCR-тексты и диагностические артефакты для следующего шага.

Для каждого запуска задаётся имя эксперимента. Его результаты сохраняются в `outputs/besy/processed_images/<эксперимент>/<имя-разворота>/`:

```text
00_left_detection_region.jpg / 00_right_detection_region.jpg
                           # широкие перекрывающиеся входы initial OCR
01_initial_boxes.jpg       # объединённые исходные боксы, seam и границы вырезов
01_initial_ocr.json        # исходные боксы и распознанный текст
02_left_crop.jpg
02_right_crop.jpg
03_left_dewarped.jpg
03_right_dewarped.jpg
04_left_boxes.jpg
04_right_boxes.jpg
left.txt / right.txt       # текст страниц в порядке чтения
left_ocr.json / right_ocr.json
metadata.json              # координаты, margin и статистика
```

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path

import cv2
import numpy as np
import paddle
from paddleocr import PaddleOCR, TextImageUnwarping


os.environ.setdefault('PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK', 'True')


PROJECT_ROOT = Path(os.environ.get(
    'SPARK_ROOT',
    '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit',
))
PAGES_DIR = PROJECT_ROOT / 'data/besy/pages'
EXPERIMENT_NAME = 'experiment_03_lower_box_threshold'
PROCESSED_ROOT = PROJECT_ROOT / 'outputs/besy/processed_images'
PROCESSED_DIR = PROCESSED_ROOT / EXPERIMENT_NAME
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SPREAD_PATHS = sorted(PAGES_DIR.glob('*.jpg'))
assert SPREAD_PATHS, f'Не найдены JPG-развороты в {PAGES_DIR}'

DEVICE = 'gpu:0' if paddle.device.cuda.device_count() else 'cpu'
print(f'Устройство PaddleOCR: {DEVICE}')
print(f'Разворотов: {len(SPREAD_PATHS)}')
print(f'Результаты: {PROCESSED_DIR}')


/home/wsl_user/miniconda3/envs/dev_env/lib/python3.12/site-packages/paddle/utils/cpp_extension/extension_utils.py:712: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


Устройство PaddleOCR: gpu:0
Разворотов: 7
Результаты: /home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/processed_images/experiment_03_lower_box_threshold


## Правило автоматического разреза

Первый проход OCR выполняется **без** dewarping на двух широких перекрывающихся половинах разворота. Это даёт детектору более крупный текст, не добавляя собственный resize и не требуя обработки целого кадра с большим лимитом памяти. Координаты боксов переносятся обратно в исходный кадр и объединяются.

Затем боксы предварительно относятся к стороне по центру изображения. Используются устойчивые края текстовых областей: 95-й перцентиль правых краёв слева и 5-й перцентиль левых краёв справа. Их середина — `seam`.

Вырезы перекрываются на `margin`: это две медианные высоты исходных OCR-боксов, но не меньше 24 пикселей. Запас сохраняет изображение строки, которую начальный OCR не нашёл у корешка; её сможет распознать OCR после dewarping. Каждый page crop всегда идёт от верхнего до нижнего края исходного кадра — OCR-боксы не могут обрезать текст сверху или снизу.

In [2]:
# PP-OCRv5 необходим для lang='ru': в PP-OCRv6 русский не поддерживается.
# BOS — запасной официальный источник моделей, если Hugging Face недоступен.
os.environ.setdefault('PADDLE_PDX_MODEL_SOURCE', 'BOS')

INITIAL_DET_LIMIT = 1280
POST_DEWARP_DET_LIMIT = 1280
POST_DEWARP_BOX_THRESH = 0.5

OCR_CONFIG = {
    'lang': 'ru',
    'ocr_version': 'PP-OCRv5',
    'device': DEVICE,
    'use_doc_orientation_classify': False,
    'use_doc_unwarping': False,  # UVDoc вызывается отдельно только после разреза.
    'use_textline_orientation': False,
    # Полный кадр 4032 px требует около 46 ГБ RAM у server-det на CPU.
    # Координаты результата PaddleOCR всё равно возвращает в масштабе исходного изображения.
    'text_det_limit_side_len': INITIAL_DET_LIMIT,
    'text_det_limit_type': 'max',
    # В текущем CPU-окружении OneDNN не совместим с PP-OCRv5; на GPU этот флаг не нужен.
    'enable_mkldnn': False if DEVICE == 'cpu' else True,
}

MARGIN_LINE_HEIGHTS = 2.0
MIN_MARGIN_PX = 24
EDGE_QUANTILE = 0.95
CROP_PADDING_LINE_HEIGHTS = 4.0
MIN_CROP_PADDING_PX = 80
INITIAL_HALF_OVERLAP_RATIO = 0.10

ocr = PaddleOCR(**OCR_CONFIG)
unwarper = TextImageUnwarping(model_name='UVDoc', device=DEVICE)
print('Модели инициализированы.')


Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/wsl_user/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('eslav_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/wsl_user/.paddlex/official_models/eslav_PP-OCRv5_mobile_rec`.
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/wsl_user/.paddlex/official_models/UVDoc`.


Модели инициализированы.


In [3]:
def to_builtin(value):
    """Преобразовать NumPy-значения в JSON-совместимые типы."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {key: to_builtin(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [to_builtin(item) for item in value]
    return value


def write_json(path: Path, payload: dict | list) -> None:
    path.write_text(
        json.dumps(to_builtin(payload), ensure_ascii=False, indent=2),
        encoding='utf-8',
    )


def first_result(prediction):
    """PaddleOCR 3.x может вернуть список или итератор из одного результата."""
    if isinstance(prediction, list):
        if len(prediction) != 1:
            raise RuntimeError(f'Ожидался один результат, получено: {len(prediction)}')
        return prediction[0]
    return next(iter(prediction))


def run_ocr(
    image_path: Path,
    det_limit_side_len: int,
    text_det_box_thresh: float | None = None,
) -> list[dict]:
    """Вернуть распознанные строки с прямоугольными координатами."""
    options = {'text_det_limit_side_len': det_limit_side_len}
    if text_det_box_thresh is not None:
        options['text_det_box_thresh'] = text_det_box_thresh
    result = first_result(ocr.predict(str(image_path), **options))
    payload = result.json['res']

    boxes = payload.get('rec_boxes', [])
    texts = payload.get('rec_texts', [])
    scores = payload.get('rec_scores', [])

    records = []
    for box, text, score in zip(boxes, texts, scores):
        text = (text or '').strip()
        if not text:
            continue
        x0, y0, x1, y1 = map(float, box)
        records.append({
            'box': [x0, y0, x1, y1],
            'text': text,
            'confidence': float(score),
        })
    return records


def translate_records(records: list[dict], dx: int, dy: int = 0) -> list[dict]:
    """Перенести координаты боксов из crop обратно в исходный разворот."""
    translated = []
    for record in records:
        x0, y0, x1, y1 = record['box']
        translated.append({
            **record,
            'box': [x0 + dx, y0 + dy, x1 + dx, y1 + dy],
        })
    return translated


def box_iou(first: dict, second: dict) -> float:
    ax0, ay0, ax1, ay1 = first['box']
    bx0, by0, bx1, by1 = second['box']
    ix0, iy0 = max(ax0, bx0), max(ay0, by0)
    ix1, iy1 = min(ax1, bx1), min(ay1, by1)
    intersection = max(0, ix1 - ix0) * max(0, iy1 - iy0)
    if not intersection:
        return 0.0
    first_area = (ax1 - ax0) * (ay1 - ay0)
    second_area = (bx1 - bx0) * (by1 - by0)
    return intersection / (first_area + second_area - intersection)


def deduplicate_records(records: list[dict], iou_threshold: float = 0.6) -> list[dict]:
    """Убрать дубликаты из перекрытия предварительных половин."""
    kept = []
    for record in sorted(records, key=lambda item: item['confidence'], reverse=True):
        if all(box_iou(record, existing) < iou_threshold for existing in kept):
            kept.append(record)
    return kept


def initial_detection_regions(image: np.ndarray) -> dict[str, tuple[int, int, int, int]]:
    """Две широкие половины с перекрытием; resize не выполняется."""
    height, width = image.shape[:2]
    half_overlap = int(round(width * INITIAL_HALF_OVERLAP_RATIO))
    midpoint = width // 2
    return {
        'left': (0, 0, min(width, midpoint + half_overlap), height),
        'right': (max(0, midpoint - half_overlap), 0, width, height),
    }


def box_height(record: dict) -> float:
    return record['box'][3] - record['box'][1]


def box_center_x(record: dict) -> float:
    return (record['box'][0] + record['box'][2]) / 2


def box_center_y(record: dict) -> float:
    return (record['box'][1] + record['box'][3]) / 2


def sort_reading_order(records: list[dict]) -> list[dict]:
    return sorted(records, key=lambda item: (box_center_y(item), item['box'][0]))


def draw_boxes(image: np.ndarray, records: list[dict], color: tuple[int, int, int]) -> np.ndarray:
    result = image.copy()
    for record in records:
        x0, y0, x1, y1 = map(lambda value: int(round(value)), record['box'])
        cv2.rectangle(result, (x0, y0), (x1, y1), color, 2)
    return result


def save_image(path: Path, image: np.ndarray) -> None:
    ok = cv2.imwrite(str(path), image)
    if not ok:
        raise RuntimeError(f'Не удалось сохранить изображение: {path}')



In [4]:
def split_spread_records(records: list[dict], image_width: int) -> tuple[list[dict], list[dict]]:
    """Предварительно разделить боксы по центру исходного кадра."""
    image_midpoint = image_width / 2
    left = [record for record in records if box_center_x(record) < image_midpoint]
    right = [record for record in records if box_center_x(record) >= image_midpoint]
    if not left or not right:
        raise RuntimeError(
            f'Не удалось выделить обе страницы: слева={len(left)}, справа={len(right)}'
        )
    return left, right


def calculate_split(left: list[dict], right: list[dict], image_width: int) -> dict:
    """Найти seam и перекрывающиеся границы вырезов."""
    left_right_edges = np.array([record['box'][2] for record in left], dtype=float)
    right_left_edges = np.array([record['box'][0] for record in right], dtype=float)
    heights = np.array([box_height(record) for record in left + right], dtype=float)

    left_text_edge = float(np.quantile(left_right_edges, EDGE_QUANTILE))
    right_text_edge = float(np.quantile(right_left_edges, 1 - EDGE_QUANTILE))
    seam = (left_text_edge + right_text_edge) / 2
    median_height = float(np.median(heights))
    margin = max(MIN_MARGIN_PX, int(round(MARGIN_LINE_HEIGHTS * median_height)))

    left_crop_end = min(image_width, int(np.ceil(seam + margin)))
    right_crop_start = max(0, int(np.floor(seam - margin)))

    return {
        'left_text_edge': left_text_edge,
        'right_text_edge': right_text_edge,
        'text_gap': right_text_edge - left_text_edge,
        'seam': seam,
        'median_box_height': median_height,
        'margin': margin,
        'left_crop_end': left_crop_end,
        'right_crop_start': right_crop_start,
    }


def crop_page(image: np.ndarray, records: list[dict], side: str, split: dict) -> tuple[np.ndarray, dict]:
    """Обрезать страницу по внешнему краю текста и с перекрытием у корешка."""
    height, width = image.shape[:2]
    median_height = float(np.median([box_height(record) for record in records]))
    padding = max(MIN_CROP_PADDING_PX, int(round(CROP_PADDING_LINE_HEIGHTS * median_height)))

    # По Y страницу никогда не обрезаем: начальный OCR может пропустить верхние строки.
    y0, y1 = 0, height

    if side == 'left':
        x0 = max(0, int(np.floor(min(record['box'][0] for record in records) - padding)))
        x1 = split['left_crop_end']
    elif side == 'right':
        x0 = split['right_crop_start']
        x1 = min(width, int(np.ceil(max(record['box'][2] for record in records) + padding)))
    else:
        raise ValueError(f'Неизвестная сторона: {side}')

    if x1 <= x0 or y1 <= y0:
        raise RuntimeError(f'Некорректная область {side}: {(x0, y0, x1, y1)}')

    crop = image[y0:y1, x0:x1]
    return crop, {'x0': x0, 'y0': y0, 'x1': x1, 'y1': y1, 'padding': padding}


def save_page_text(folder: Path, side: str, records: list[dict]) -> None:
    ordered = sort_reading_order(records)
    text = '\n'.join(record['text'] for record in ordered)
    (folder / f'{side}.txt').write_text(text + ('\n' if text else ''), encoding='utf-8')
    write_json(folder / f'{side}_ocr.json', {
        'line_count': len(ordered),
        'mean_confidence': float(np.mean([record['confidence'] for record in ordered])) if ordered else None,
        'records': ordered,
    })


def unwarp_image(source_path: Path, destination_path: Path) -> None:
    result = first_result(unwarper.predict(str(source_path)))
    dewarped = np.asarray(result['doctr_img'])
    save_image(destination_path, dewarped)


In [5]:
def process_spread(spread_path: Path) -> dict:
    folder = PROCESSED_DIR / spread_path.stem
    folder.mkdir(parents=True, exist_ok=True)

    image = cv2.imread(str(spread_path))
    if image is None:
        raise RuntimeError(f'Не удалось прочитать {spread_path}')

    detection_regions = initial_detection_regions(image)
    region_records = {}
    for side, (x0, y0, x1, y1) in detection_regions.items():
        region_path = folder / f'00_{side}_detection_region.jpg'
        save_image(region_path, image[y0:y1, x0:x1])
        region_records[side] = translate_records(run_ocr(region_path, INITIAL_DET_LIMIT), x0, y0)

    raw_records = deduplicate_records(region_records['left'] + region_records['right'])
    if not raw_records:
        raise RuntimeError(f'На развороте не найдено OCR-боксов: {spread_path.name}')

    left_initial, right_initial = split_spread_records(raw_records, image.shape[1])
    split = calculate_split(left_initial, right_initial, image.shape[1])

    initial_overlay = image.copy()
    for record in left_initial:
        x0, y0, x1, y1 = map(lambda value: int(round(value)), record['box'])
        cv2.rectangle(initial_overlay, (x0, y0), (x1, y1), (255, 0, 0), 2)
    for record in right_initial:
        x0, y0, x1, y1 = map(lambda value: int(round(value)), record['box'])
        cv2.rectangle(initial_overlay, (x0, y0), (x1, y1), (0, 0, 255), 2)
    seam = int(round(split['seam']))
    cv2.line(initial_overlay, (seam, 0), (seam, image.shape[0]), (0, 255, 0), 3)
    cv2.line(initial_overlay, (split['left_crop_end'], 0), (split['left_crop_end'], image.shape[0]), (0, 255, 255), 2)
    cv2.line(initial_overlay, (split['right_crop_start'], 0), (split['right_crop_start'], image.shape[0]), (0, 255, 255), 2)
    save_image(folder / '01_initial_boxes.jpg', initial_overlay)
    write_json(folder / '01_initial_ocr.json', {
        'detection_regions': detection_regions,
        'region_records': region_records,
        'left_records': left_initial,
        'right_records': right_initial,
    })

    left_crop, left_crop_rect = crop_page(image, left_initial, 'left', split)
    right_crop, right_crop_rect = crop_page(image, right_initial, 'right', split)
    left_crop_path = folder / '02_left_crop.jpg'
    right_crop_path = folder / '02_right_crop.jpg'
    save_image(left_crop_path, left_crop)
    save_image(right_crop_path, right_crop)

    left_dewarped_path = folder / '03_left_dewarped.jpg'
    right_dewarped_path = folder / '03_right_dewarped.jpg'
    unwarp_image(left_crop_path, left_dewarped_path)
    unwarp_image(right_crop_path, right_dewarped_path)

    left_dewarped = cv2.imread(str(left_dewarped_path))
    right_dewarped = cv2.imread(str(right_dewarped_path))
    left_records = run_ocr(
        left_dewarped_path,
        POST_DEWARP_DET_LIMIT,
        POST_DEWARP_BOX_THRESH,
    )
    right_records = run_ocr(
        right_dewarped_path,
        POST_DEWARP_DET_LIMIT,
        POST_DEWARP_BOX_THRESH,
    )
    save_image(folder / '04_left_boxes.jpg', draw_boxes(left_dewarped, left_records, (255, 0, 0)))
    save_image(folder / '04_right_boxes.jpg', draw_boxes(right_dewarped, right_records, (0, 0, 255)))
    save_page_text(folder, 'left', left_records)
    save_page_text(folder, 'right', right_records)

    metadata = {
        'experiment_name': EXPERIMENT_NAME,
        'source_image': str(spread_path),
        'source_shape': list(image.shape),
        'initial_detection_regions': detection_regions,
        'split': split,
        'left_crop_rect': left_crop_rect,
        'right_crop_rect': right_crop_rect,
        'initial_box_counts': {'left': len(left_initial), 'right': len(right_initial)},
        'post_dewarping_ocr': {
            'det_limit_side_len': POST_DEWARP_DET_LIMIT,
            'text_det_box_thresh': POST_DEWARP_BOX_THRESH,
        },
        'dewarped_box_counts': {'left': len(left_records), 'right': len(right_records)},
    }
    write_json(folder / 'metadata.json', metadata)

    return {
        'spread': spread_path.name,
        'folder': str(folder),
        'seam': round(split['seam'], 1),
        'margin': split['margin'],
        'initial_left_boxes': len(left_initial),
        'initial_right_boxes': len(right_initial),
        'left_lines_after_dewarping': len(left_records),
        'right_lines_after_dewarping': len(right_records),
    }


In [6]:
summaries = []
for spread_path in SPREAD_PATHS:
    print(f'Обработка: {spread_path.name}')
    summary = process_spread(spread_path)
    summaries.append(summary)
    print(
        f"  seam={summary['seam']}, margin={summary['margin']}, "
        f"строк после dewarping: {summary['left_lines_after_dewarping']} / "
        f"{summary['right_lines_after_dewarping']}"
    )

write_json(PROCESSED_DIR / 'summary.json', summaries)
summaries


Обработка: 10-11.jpg
  seam=2006.8, margin=134, строк после dewarping: 41 / 38
Обработка: 182-183.jpg
  seam=1985.5, margin=178, строк после dewarping: 40 / 38
Обработка: 356-357.jpg
  seam=2156.2, margin=162, строк после dewarping: 39 / 41
Обработка: 528-529.jpg
  seam=2038.6, margin=220, строк после dewarping: 42 / 40
Обработка: 614-615.jpg
  seam=2211.4, margin=154, строк после dewarping: 40 / 40
Обработка: 698-699.jpg
  seam=2003.5, margin=132, строк после dewarping: 40 / 40
Обработка: 96-97.jpg
  seam=2049.3, margin=194, строк после dewarping: 38 / 43


[{'spread': '10-11.jpg',
  'folder': '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/processed_images/experiment_03_lower_box_threshold/10-11',
  'seam': 2006.8,
  'margin': 134,
  'initial_left_boxes': 70,
  'initial_right_boxes': 72,
  'left_lines_after_dewarping': 41,
  'right_lines_after_dewarping': 38},
 {'spread': '182-183.jpg',
  'folder': '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/processed_images/experiment_03_lower_box_threshold/182-183',
  'seam': 1985.5,
  'margin': 178,
  'initial_left_boxes': 71,
  'initial_right_boxes': 60,
  'left_lines_after_dewarping': 40,
  'right_lines_after_dewarping': 38},
 {'spread': '356-357.jpg',
  'folder': '/home/wsl_user/my_projects/SPARK — Synchronized Print-Audio Reading Kit/outputs/besy/processed_images/experiment_03_lower_box_threshold/356-357',
  'seam': 2156.2,
  'margin': 162,
  'initial_left_boxes': 63,
  'initial_right_boxes': 68,
  'left_lines_after_de

## Проверка результата

Для каждого каталога последовательно проверить:

1. `01_initial_boxes.jpg`: синие боксы — левая страница, красные — правая, зелёная линия — `seam`, жёлтые — границы перекрывающихся вырезов.
2. `02_*_crop.jpg`: обе страницы целиком сохранились; у корешка есть небольшое перекрытие.
3. `03_*_dewarped.jpg`: UVDoc не внёс заметных артефактов.
4. `04_*_boxes.jpg` и `left.txt` / `right.txt`: строки после dewarping читаются и принадлежат правильной странице.

Если разворот не проходит эти визуальные проверки, он не используется для следующего шага matching и создания anchor.